# Titanic Survival Prediction with a Decision Tree

This Colab-ready notebook builds a **classification** model for Kaggle's Titanic competition. It uses passenger details to predict `Survived` (0 = no, 1 = yes). The workflow follows the linked Kaggle decision-tree tutorial: inspect the data, prepare features, fit and visualize a tree, validate it, then optionally create a Kaggle submission.

**Run all cells from top to bottom.** If you have the competition files, upload `train.csv` before running the data-loading cell and also upload `test.csv` before the final optional submission cell. Otherwise, the notebook loads a public copy of the labeled training data for the demonstration and reports validation results.

Sources: [Kaggle tutorial notebook](https://www.kaggle.com/code/hamelg/python-for-data-29-decision-trees) · [Titanic competition data and rules](https://www.kaggle.com/competitions/titanic/data)


In [1]:
import pandas as pd

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

print("Pandas version:", pd.__version__)


Pandas version: 3.0.1


## 1. Load the Titanic data

The official competition files are named `train.csv` and `test.csv`. Join the competition and accept its rules to download them. If `train.csv` is present in the Colab session, this notebook uses it; otherwise it fetches a public copy of the labeled training data for a reproducible demonstration. The public copy is for validation practice; upload Kaggle's official `test.csv` to create a competition submission.

In [2]:
train_path = Path("train.csv")
if train_path.exists():
    train = pd.read_csv(train_path)
    data_source = "uploaded train.csv"
else:
    data_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    train = pd.read_csv(data_url)
    data_source = "public Titanic training data mirror"

print("Data source:", data_source)
print("Rows and columns:", train.shape)
print("Columns:", list(train.columns))
display(train.head())


Data source: uploaded train.csv
Rows and columns: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
 PassengerId  Survived  Pclass                                                Name    Sex  Age  SibSp  Parch           Ticket    Fare Cabin Embarked
           1         0       3                             Braund, Mr. Owen Harris   male 22.0      1      0        A/5 21171  7.2500   NaN        S
           2         1       1 Cumings, Mrs. John Bradley (Florence Briggs Thayer) female 38.0      1      0         PC 17599 71.2833   C85        C
           3         1       3                              Heikkinen, Miss. Laina female 26.0      0      0 STON/O2. 3101282  7.9250   NaN        S
           4         1       1        Futrelle, Mrs. Jacques Heath (Lily May Peel) female 35.0      1      0           113803 53.1000  C123        S
           5         0       3                            Allen, Mr. 

## 2. Explore the target and missing values

`Survived` is the value the model learns to predict. Check how many passengers survived and where data is missing before training.

In [3]:
print("Survival counts (0 = did not survive, 1 = survived):")
print(train["Survived"].value_counts().sort_index())
print("\nMissing values in selected columns:")
selected_columns = ["Survived", "Sex", "Pclass", "Age", "SibSp", "Parch", "Fare", "Embarked"]
print(train[selected_columns].isna().sum())


Survival counts (0 = did not survive, 1 = survived):
Survived
0    549
1    342
Name: count, dtype: int64

Missing values in selected columns:
Survived      0
Sex           0
Pclass        0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64


## 3. Prepare features and split a validation set

We keep a portion of the labeled data aside to estimate performance on passengers the model did not train on. The preprocessing is part of a scikit-learn pipeline, so missing-value handling and category encoding are learned using only the training split.

In [4]:
features = ["Sex", "Pclass", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = train[features]
y = train["Survived"]

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))


Training rows: 712
Validation rows: 179


## 4. Train a decision tree

A decision tree learns a series of feature-based rules. Limiting its depth and requiring a minimum number of rows per leaf helps control complexity and reduce overfitting.

In [5]:
model = Pipeline(steps=[
    ("prepare", preprocessor),
    ("tree", DecisionTreeClassifier(
        criterion="gini",
        max_depth=4,
        min_samples_leaf=5,
        random_state=42,
    )),
])

model.fit(X_train, y_train)
valid_predictions = model.predict(X_valid)

print(f"Validation accuracy: {accuracy_score(y_valid, valid_predictions):.3f}")
print("\nConfusion matrix (actual rows, predicted columns):")
print(confusion_matrix(y_valid, valid_predictions))
print("\nClassification report:")
print(classification_report(y_valid, valid_predictions, target_names=["Did not survive", "Survived"], zero_division=0))


Validation accuracy: 0.777

Confusion matrix (actual rows, predicted columns):
[[101   9]
 [ 31  38]]

Classification report:
                 precision    recall  f1-score   support

Did not survive       0.77      0.92      0.83       110
       Survived       0.81      0.55      0.66        69

       accuracy                           0.78       179
      macro avg       0.79      0.73      0.74       179
   weighted avg       0.78      0.78      0.77       179



## 5. Inspect the learned rules

The text view shows the if/then splits in the fitted tree. A split such as `numeric__Pclass <= 2.5` divides passengers into groups based on ticket class.

In [6]:
encoded_feature_names = model.named_steps["prepare"].get_feature_names_out()
tree_text = export_text(
    model.named_steps["tree"],
    feature_names=list(encoded_feature_names),
    max_depth=4,
)
print(tree_text)


|--- categorical__Sex_female <= 0.50
|   |--- numeric__Age <= 3.50
|   |   |--- numeric__Pclass <= 2.50
|   |   |   |--- class: 1
|   |   |--- numeric__Pclass >  2.50
|   |   |   |--- class: 1
|   |--- numeric__Age >  3.50
|   |   |--- numeric__Pclass <= 1.50
|   |   |   |--- numeric__Fare <= 26.14
|   |   |   |   |--- class: 0
|   |   |   |--- numeric__Fare >  26.14
|   |   |   |   |--- class: 0
|   |   |--- numeric__Pclass >  1.50
|   |   |   |--- numeric__Fare <= 51.70
|   |   |   |   |--- class: 0
|   |   |   |--- numeric__Fare >  51.70
|   |   |   |   |--- class: 0
|--- categorical__Sex_female >  0.50
|   |--- numeric__Pclass <= 2.50
|   |   |--- numeric__Fare <= 28.86
|   |   |   |--- numeric__Age <= 37.00
|   |   |   |   |--- class: 1
|   |   |   |--- numeric__Age >  37.00
|   |   |   |   |--- class: 1
|   |   |--- numeric__Fare >  28.86
|   |   |   |--- numeric__Parch <= 1.50
|   |   |   |   |--- class: 1
|   |   |   |--- numeric__Parch >  1.50
|   |   |   |   |--- class: 1
|  

## 6. Rank the features

Feature importance summarizes how much each encoded feature contributed to the tree's splits. It is a useful diagnostic, but it does not prove that a feature causes survival.

In [7]:
importance = pd.DataFrame({
    "Feature": encoded_feature_names,
    "Importance": model.named_steps["tree"].feature_importances_,
}).sort_values("Importance", ascending=False)
print(importance.to_string(index=False))


                Feature  Importance
categorical__Sex_female    0.593517
        numeric__Pclass    0.206451
           numeric__Age    0.105275
          numeric__Fare    0.055615
categorical__Embarked_S    0.038309
         numeric__Parch    0.000833
         numeric__SibSp    0.000000
  categorical__Sex_male    0.000000
categorical__Embarked_C    0.000000
categorical__Embarked_Q    0.000000


## 7. Optional: create a Kaggle submission

Upload the official competition `test.csv` to the Colab session before running this cell. The test file has no `Survived` column, so the model predicts it. The saved submission has only `PassengerId` and `Survived`, as required by the competition.

In [8]:
test_path = Path("test.csv")
if test_path.exists():
    test = pd.read_csv(test_path)
    model.fit(X, y)  # Refit on all labeled training rows
    test_predictions = model.predict(test[features])

    submission = pd.DataFrame({
        "PassengerId": test["PassengerId"],
        "Survived": test_predictions.astype(int),
    })
    submission.to_csv("submission.csv", index=False)
    print("Saved submission.csv")
    display(submission.head())

    try:
        from google.colab import files
        files.download("submission.csv")
    except ImportError:
        print("Download submission.csv from the notebook's file browser.")
else:
    print("No official test.csv found. Validation results above are ready; upload test.csv to generate submission.csv.")


No official test.csv found. Validation results above are ready; upload test.csv to generate submission.csv.


## Conclusion

This notebook prepares mixed numeric/categorical data, fits a decision tree, evaluates it on held-out passengers, displays the rules and feature importances, and can create a Kaggle submission when the official test file is provided. Try changing `max_depth` and `min_samples_leaf`, then compare validation accuracy to see how tree complexity affects generalization.
